In [0]:
%sql
CREATE TABLE IF NOT EXISTS de_learning_workspace.gold.dim_customer
(
    customer_sk BIGINT GENERATED ALWAYS AS IDENTITY,

    customer_id INT,

    first_name STRING,
    last_name STRING,
    email STRING,
    phone STRING,

    city STRING,
    state STRING,
    country STRING,

    effective_start_date TIMESTAMP,
    effective_end_date TIMESTAMP,

    is_current BOOLEAN,

    created_timestamp TIMESTAMP,
    updated_timestamp TIMESTAMP
)
USING DELTA;

In [0]:
# %sql

# INSERT INTO de_learning_workspace.gold.dim_customer
# (
#     customer_id,
#     first_name,
#     last_name,
#     email,
#     phone,
#     city,
#     state,
#     country,
#     effective_start_date,
#     effective_end_date,
#     is_current,
#     created_timestamp,
#     updated_timestamp
# )

# SELECT

#     customer_id,
#     first_name,
#     last_name,
#     email,
#     phone,
#     city,
#     state,
#     country,

#     current_timestamp(),
#     NULL,
#     TRUE,

#     created_at,
#     updated_at

# FROM de_learning_workspace.silver.customers;

In [0]:
# (
#     dim_customer_df
#     .write
#     .format("delta")
#     .mode("overwrite")
#     .saveAsTable(
#         "de_learning_workspace.gold.dim_customer"
#     )
# )

In [0]:
# %sql
# select * from de_learning_workspace.gold.dim_customer

In [0]:
%sql
UPDATE de_learning_workspace.gold.dim_customer AS target
SET
    target.is_current = false,
    target.effective_end_date = current_timestamp()
WHERE target.is_current = true
  AND EXISTS (
      SELECT 1
      FROM de_learning_workspace.silver.customers AS source
      WHERE source.customer_id = target.customer_id
        AND (
             COALESCE(source.city, '') <> COALESCE(target.city, '')
          OR COALESCE(source.state, '') <> COALESCE(target.state, '')
          OR COALESCE(source.country, '') <> COALESCE(target.country, '')
        )
  );

In [0]:
%sql
INSERT INTO de_learning_workspace.gold.dim_customer
(
    customer_id,
    first_name,
    last_name,
    email,
    phone,
    city,
    state,
    country,
    effective_start_date,
    effective_end_date,
    is_current,
    created_timestamp,
    updated_timestamp
)

select
    s.customer_id,
    s.first_name,
    s.last_name,
    s.email,
    s.phone,
    s.city,
    s.state,
    s.country,
    current_timestamp(),
    null,
    true,
    s.created_at,
    s.updated_at

from de_learning_workspace.silver.customers as s where not EXISTS (select 1 from de_learning_workspace.gold.dim_customer t where t.customer_id=s.customer_id and t.is_current=true
)

In [0]:
print("Gold Layer Dim Customer Loaded Successfully")

In [0]:
%sql
select * from de_learning_workspace.gold.dim_customer